# Case Study: Betti-0 Trajectories Around Representative Events

Supplementary figures for reviewer response.

Four events covering different mechanistic categories:
- **Market Milestone**: BTC Bull Market Peak (2021-04-10)
- **Regulation**: China Comprehensive Crypto Ban (2021-09-24)
- **Systemic Failure**: Terra/LUNA Collapse (onset 2022-05-01)
- **Systemic Failure**: FTX Collapse (2022-11-01)

Each panel shows log-DTW Betti-0 (top) vs VIX & BVOL (bottom) in a ±35 day window, with the pre-event window [−21, 0] shaded.

In [ ]:
from pathlib import Path
from datetime import timedelta
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

DATA_PATH = Path('/Users/jane/Documents/202511吾-Systems/3.Data/'
                 'TDA_SP500_VIX_BTC_BVOL_CVI_merged0329_labeled_with_network.csv')
DATA_OUT  = Path('/Users/jane/Documents/202511吾-Systems/12.Data0329')

df = pd.read_csv(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)
for c in ['VIX','BVOL','CVI','SP500']:
    df[c] = df[c].ffill()

CASES = [
    {
        'label':    'BTC Bull Market Peak',
        'category': 'Market Milestone',
        'anchor':   '2021-04-10',
        'related':  [('2021-01-02','BTC $30k'),('2021-01-07','BTC $40k'),
                     ('2021-02-16','BTC $50k'),('2021-03-13','BTC $60k')],
        'color':    '#4DBBD5',
        'note':     'Betti-0 rises steadily in the 30 days before the cycle peak,\n'
                    'reaching +2.5 s.d. above baseline — network fragmentation as\n'
                    'capital rotates selectively (late-stage bull-market topology).\n'
                    'VIX declines concurrently, confirming low macro-risk perception.',
    },
    {
        'label':    'China Comprehensive Crypto Ban',
        'category': 'Regulation',
        'anchor':   '2021-09-24',
        'related':  [('2021-05-19','China crackdown (May)')],
        'color':    '#E64B35',
        'note':     'Betti-0 shows no systematic pre-event elevation (mean z ≈ +0.19).\n'
                    'Regulatory shocks are largely unanticipated in cross-asset topology\n'
                    'because their impact depends on enforcement credibility rather\n'
                    'than gradual structural buildup. VIX also remains stable pre-event.',
    },
    {
        'label':    'Terra / LUNA Collapse',
        'category': 'Systemic Failure',
        'anchor':   '2022-05-01',
        'related':  [('2022-05-11','UST collapse'),('2022-05-12','Terra collapse'),
                     ('2022-05-13','Luna delisting')],
        'color':    '#F39B7F',
        'note':     'Betti-0 contracts in the 3 weeks before depeg onset (z ≈ −0.23),\n'
                    'consistent with pre-crisis network consolidation. A sharp Betti-0\n'
                    'spike occurs at collapse onset (day 0–5) as structure fragments\n'
                    'abruptly. VIX rises concurrently, confirming macro-fear spillover.',
    },
    {
        'label':    'FTX Collapse',
        'category': 'Systemic Failure',
        'anchor':   '2022-11-01',
        'related':  [],
        'color':    '#8B4513',
        'note':     'Similar pre-event pattern to Terra/LUNA: Betti-0 below baseline\n'
                    'in [−21, −1] (z ≈ −0.15), then sharply reverses post-event.\n'
                    'VIX surges strongly at event onset, capturing contagion spillover\n'
                    'to traditional markets not pre-empted by topological signal alone.',
    },
]

PRE  = 35
POST = 15

def get_window(anchor_str):
    anchor = pd.to_datetime(anchor_str)
    mask = ((df['date'] >= anchor - timedelta(days=PRE)) &
            (df['date'] <= anchor + timedelta(days=POST)))
    sub = df[mask].copy()
    sub['day'] = (sub['date'] - anchor).dt.days
    return sub, anchor

def zs(s):
    mu, sd = s.mean(), s.std()
    return (s - mu) / sd if sd > 0 else s - mu

print(f'Dataset: {len(df)} rows, {df.date.min().date()} to {df.date.max().date()}')


In [ ]:
# ── Figure 1: 4-column overview ──────────────────────────────────────────────
fig = plt.figure(figsize=(18, 8.5))
outer = gridspec.GridSpec(2, 4, figure=fig,
                          height_ratios=[3.2, 2.4],
                          hspace=0.10, wspace=0.32)

for col_idx, case in enumerate(CASES):
    sub, anchor = get_window(case['anchor'])
    color = case['color']

    ax0 = fig.add_subplot(outer[0, col_idx])
    ax1 = fig.add_subplot(outer[1, col_idx])

    bz    = zs(sub['log_dtw_betti0'])
    bz_rm = bz.rolling(5, center=True, min_periods=1).mean()
    vz    = zs(sub['VIX'])
    bvz   = zs(sub['BVOL'])

    for ax in [ax0, ax1]:
        ax.axvspan(-21, 0, color='#eaf4fb', alpha=0.55, zorder=0)
        ax.axhline(0, color='grey', lw=0.7, ls='--', alpha=0.5)
        ax.axvline(0, color=color, lw=2.0, ls='-', alpha=0.85, zorder=4)
        for rel_date, _ in case['related']:
            rd = (pd.to_datetime(rel_date) - anchor).days
            if -PRE <= rd <= POST:
                ax.axvline(rd, color=color, lw=0.9, ls=':', alpha=0.6)
        ax.set_xlim(-PRE, POST)
        ax.yaxis.grid(True, ls='--', alpha=0.25)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.xaxis.set_visible(False)

    ax0.plot(sub['day'], bz,    color=color, lw=2.2, zorder=5)
    ax0.plot(sub['day'], bz_rm, color=color, lw=1.1, ls='--', alpha=0.65, zorder=6)
    ax0.fill_between(sub['day'], bz, 0,
                     where=(sub['day'] >= -21) & (sub['day'] <= 0),
                     alpha=0.18, color=color, zorder=3)
    ax0.set_ylabel('log-DTW Betti-0\n(z-score)', fontsize=8.5)
    ax0.text(0.97, 0.96, case['category'], transform=ax0.transAxes,
             ha='right', va='top', fontsize=7.5, fontweight='bold', color='white',
             bbox=dict(boxstyle='round,pad=0.3', facecolor=color, edgecolor='none'))
    ax0.set_title(case['label'], fontsize=11, fontweight='bold', pad=6)

    ax1.plot(sub['day'], vz,  color='#3C5488', lw=1.8, ls='-',  zorder=5)
    ax1.plot(sub['day'], bvz, color='#DC0000', lw=1.8, ls='--', zorder=5)
    ax1.set_ylabel('VIX / BVOL\n(z-score)', fontsize=8.5)
    ax1.xaxis.set_visible(True)
    ax1.set_xlabel('Days relative to event', fontsize=8.5)
    ax1.spines['bottom'].set_visible(True)

# ── Shared legend ─────────────────────────────────────────────────────────────
legend_elements = [
    # Topological signal (row 1)
    Line2D([0], [0], color='#555555', lw=2.2,
           label='log-DTW Betti-0 (z-score)'),
    Line2D([0], [0], color='#555555', lw=1.1, ls='--', alpha=0.75,
           label='5-day rolling mean (Betti-0)'),
    # Financial baselines (row 2)
    Line2D([0], [0], color='#3C5488', lw=1.8, ls='-',
           label='VIX (z-score)'),
    Line2D([0], [0], color='#DC0000', lw=1.8, ls='--',
           label='BVOL (z-score)'),
    # Shading / event markers (row 3)
    mpatches.Patch(facecolor='#eaf4fb', edgecolor='#88aacc', alpha=0.9,
                   label='Pre-event window [−21, 0]'),
    Line2D([0], [0], color='#888888', lw=2.0, ls='-',
           label='Event date (day 0)  [line color = category]'),
    Line2D([0], [0], color='#888888', lw=0.9, ls=':',
           label='Related event (dotted vertical line)'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=4,
           fontsize=9.5, frameon=True, framealpha=0.92,
           edgecolor='#cccccc', bbox_to_anchor=(0.5, -0.04))

fig.suptitle(
    'Case Studies: Topological Betti-0 Signal vs Financial Baselines\n'
    'Around Representative Events of Different Mechanistic Categories',
    fontsize=13.5, fontweight='bold', y=1.01)

fig.savefig(DATA_OUT / 'fig_case_study_betti0_vs_baselines.png', dpi=300, bbox_inches='tight')
fig.savefig(DATA_OUT / 'fig_case_study_betti0_vs_baselines.pdf', bbox_inches='tight')
plt.show()
print('Figure 1 (overview) saved.')


In [ ]:
# ── Figure 2: individual high-resolution panels ──────────────────────────────
for case in CASES:
    sub, anchor = get_window(case['anchor'])
    color = case['color']
    fn = case['anchor'].replace('-','') + '_' + case['category'].replace(' ','_').replace('/','')

    fig = plt.figure(figsize=(10, 7))
    gs  = gridspec.GridSpec(3, 1, figure=fig,
                            height_ratios=[2.8, 2.2, 1.2], hspace=0.08)
    ax0 = fig.add_subplot(gs[0])
    ax1 = fig.add_subplot(gs[1])
    ax2 = fig.add_subplot(gs[2])

    bz  = zs(sub['log_dtw_betti0'])
    vz  = zs(sub['VIX'])
    bvz = zs(sub['BVOL'])

    for ax in [ax0, ax1]:
        ax.axvspan(-21, 0, color='#eaf4fb', alpha=0.55, zorder=0)
        ax.axhline(0, color='grey', lw=0.7, ls='--', alpha=0.5)
        ax.axvline(0, color=color, lw=2.2, ls='-', alpha=0.9, zorder=4)
        for rel_date, rel_name in case['related']:
            rd = (pd.to_datetime(rel_date) - anchor).days
            if -PRE <= rd <= POST:
                ax.axvline(rd, color=color, lw=1.0, ls=':', alpha=0.55)
                if ax is ax0:
                    ax.text(rd, bz.max()*0.85, rel_name, fontsize=6.5,
                            rotation=90, va='top', ha='right', color=color, alpha=0.75)
        ax.set_xlim(-PRE, POST)
        ax.yaxis.grid(True, ls='--', alpha=0.22)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.xaxis.set_visible(False)

    ax0.plot(sub['day'], bz, color=color, lw=2.4, zorder=5, label='log-DTW Betti-0')
    ax0.fill_between(sub['day'], bz, 0,
                     where=(sub['day']>=-21)&(sub['day']<=0),
                     alpha=0.20, color=color, zorder=3)
    ax0.plot(sub['day'], bz.rolling(5,center=True,min_periods=1).mean(),
             color=color, lw=1.0, ls='--', alpha=0.55)
    ax0.set_ylabel('log-DTW Betti-0\n(z-score)', fontsize=9.5)
    ax0.legend(frameon=False, fontsize=9, loc='upper left')
    ax0.text(0.99, 0.97, case['category'], transform=ax0.transAxes,
             ha='right', va='top', fontsize=8.5, fontweight='bold', color='white',
             bbox=dict(boxstyle='round,pad=0.35', facecolor=color, edgecolor='none'))

    ax1.plot(sub['day'], vz,  color='#3C5488', lw=1.9, ls='-',  label='VIX',  zorder=5)
    ax1.plot(sub['day'], bvz, color='#E64B35', lw=1.9, ls='--', label='BVOL', zorder=5)
    ax1.set_ylabel('VIX / BVOL\n(z-score)', fontsize=9.5)
    ax1.legend(frameon=False, fontsize=9, loc='upper left')
    ax1.xaxis.set_visible(True)
    ax1.set_xlabel('Days relative to event date', fontsize=10)
    ax1.spines['bottom'].set_visible(True)

    ax2.axis('off')
    ax2.text(0.5, 0.5, case['note'], transform=ax2.transAxes,
             ha='center', va='center', fontsize=9, style='italic', color='#2a2a2a',
             bbox=dict(boxstyle='round,pad=0.5', facecolor='#f5f5f5',
                       edgecolor='#cccccc', lw=0.8))

    fig.suptitle(case['label'], fontsize=13, fontweight='bold')
    fig.tight_layout()
    fig.savefig(DATA_OUT/f'fig_case_study_{fn}.png', dpi=300, bbox_inches='tight')
    fig.savefig(DATA_OUT/f'fig_case_study_{fn}.pdf', bbox_inches='tight')
    plt.show()
    print(f'Saved: fig_case_study_{fn}.png')


In [ ]:
# ── Export window data ────────────────────────────────────────────────────────
rows = []
for case in CASES:
    sub, anchor = get_window(case['anchor'])
    for _, row in sub.iterrows():
        rows.append({'case': case['label'], 'category': case['category'],
                     'anchor': case['anchor'], 'date': row['date'].date(),
                     'day': int((row['date'] - anchor).days),
                     'log_dtw_betti0': row['log_dtw_betti0'],
                     'log_wass_betti0': row['log_wass_betti0'],
                     'vol_dtw_betti0': row['vol_dtw_betti0'],
                     'VIX': row['VIX'], 'BVOL': row['BVOL'], 'CVI': row['CVI']})
pd.DataFrame(rows).to_csv(DATA_OUT/'tbl_case_study_window_data.csv', index=False)
print('Window data saved.')
